## Conf and Dep

In [ ]:
from pathlib import Path
from time import perf_counter
import json
import platform
import re
import unicodedata
import joblib
import numpy as np
import pandas as pd
import sklearn
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neural_network import MLPClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value.to_string() if isinstance(value, (pd.DataFrame, pd.Series)) else value)

SEED = 42
TEST_SIZE = 0.20
VALIDATION_SIZE = 0.20
MAX_FEATURES = 40_000
BATCH_SIZE = 256
MAX_ITER = 35
EARLY_STOPPING = True
N_ITER_NO_CHANGE = 5

FEATURES = {
    'word_1_2': {'analyzer': 'word', 'ngram_range': (1, 2)},
    'char_wb_3_5': {'analyzer': 'char_wb', 'ngram_range': (3, 5)},
}

ARCHITECTURES = {
    'hidden_32': (32,),
    'hidden_64': (64,),
}

LABEL_NAMES = ['0: nonjudi (asumsi)', '1: promosi judi (asumsi)']

PROJECT_DIR = Path.cwd().resolve()
if not (PROJECT_DIR / 'dataset').is_dir():
    PROJECT_DIR = PROJECT_DIR.parent

CLEAN_PATH = PROJECT_DIR / 'outputs/cleaning/komentar_cleaned.csv'
MANIFEST_PATH = PROJECT_DIR / 'outputs/bayesian/original_split_manifest.csv'
SYNTHETIC_PATH = PROJECT_DIR / 'outputs/augmentation/komentar_synthetic_only.csv'
OUT_DIR = PROJECT_DIR / 'outputs/ann'

pd.set_option('display.max_colwidth', 120)
print('Python:', platform.python_version(), '| scikit-learn:', sklearn.__version__)

## 2. Load data

In [ ]:
for path in [CLEAN_PATH, MANIFEST_PATH, SYNTHETIC_PATH]:
    if not path.is_file():
        raise FileNotFoundError(f'{path} belum tersedia.')

original = pd.read_csv(CLEAN_PATH, dtype=str, keep_default_na=False, encoding='utf-8-sig')
manifest = pd.read_csv(MANIFEST_PATH, dtype=str, keep_default_na=False)
synthetic = pd.read_csv(SYNTHETIC_PATH, dtype=str, keep_default_na=False, encoding='utf-8-sig')

assert original['source_row'].is_unique
assert manifest['source_row'].is_unique
assert set(original['source_row']) == set(manifest['source_row'])
assert original['label'].isin(['0', '1']).all()
assert synthetic['label'].eq('1').all()
assert synthetic['is_synthetic'].str.lower().eq('true').all()
assert not original['komentar_processed'].duplicated().any()
assert not synthetic['komentar_processed'].duplicated().any()
assert set(original['komentar_processed']).isdisjoint(synthetic['komentar_processed'])

original['label'] = original['label'].astype(int)
synthetic['label'] = synthetic['label'].astype(int)

data = manifest.merge(original, on='source_row', how='left', validate='one_to_one')
data['label'] = data['label'].astype(int)

print(f'Data asli: {len(original):,}; sintetis: {len(synthetic):,}; manifest: {len(manifest):,}')
display(data.head(3))

## 3. Data split

In [ ]:
train = data.loc[data['split'].eq('train')].copy().reset_index(drop=True)
validation = data.loc[data['split'].eq('validation')].copy().reset_index(drop=True)
test = data.loc[data['split'].eq('test')].copy().reset_index(drop=True)

assert (len(train), len(validation), len(test)) == (5360, 1787, 1787)
assert len(synthetic) == 1066

parts = {'train': train, 'validation': validation, 'test': test}
for left, right in [('train', 'validation'), ('train', 'test'), ('validation', 'test')]:
    assert set(parts[left]['source_row']).isdisjoint(parts[right]['source_row'])
    assert set(parts[left]['komentar_processed']).isdisjoint(parts[right]['komentar_processed'])

split_counts = pd.DataFrame({name: part['label'].value_counts() for name, part in parts.items()}).T.fillna(0).astype(int)
split_counts['total'] = split_counts.sum(axis=1)
display(split_counts)
print('Tambahan komentar sintetis:', len(synthetic))

## 4. Pipeline dan matriks evaluasi

In [ ]:
def make_pipeline(feature_name, hidden_layer_sizes):
    return Pipeline([
        ('tfidf', TfidfVectorizer(
            **FEATURES[feature_name],
            lowercase=False,
            min_df=2,
            max_features=MAX_FEATURES,
            sublinear_tf=True,
        )),
        ('ann', MLPClassifier(
            hidden_layer_sizes=hidden_layer_sizes,
            activation='relu',
            solver='adam',
            alpha=0.0001,
            batch_size=BATCH_SIZE,
            learning_rate_init=0.001,
            max_iter=MAX_ITER,
            early_stopping=EARLY_STOPPING,
            n_iter_no_change=N_ITER_NO_CHANGE,
            random_state=SEED,
        )),
    ])

def metrics(y_true, y_pred):
    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'precision_1': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall_1': float(recall_score(y_true, y_pred, zero_division=0)),
        'f1_1': float(f1_score(y_true, y_pred, zero_division=0)),
        'macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
    }

common_cols = [c for c in train.columns if c in synthetic.columns]
training_sets = {
    'original_only': train,
    'with_synthetic': pd.concat([train[common_cols], synthetic[common_cols]], ignore_index=True),
}

## 5. Training kandidat lewat validation

In [ ]:
tuning_records = []
for arm, fit_data in training_sets.items():
    for feat_key in FEATURES:
        for arch_key, hidden_sizes in ARCHITECTURES.items():
            candidate = make_pipeline(feat_key, hidden_sizes)
            start = perf_counter()
            candidate.fit(fit_data['komentar_processed'], fit_data['label'])
            fit_seconds = perf_counter() - start
            pred = candidate.predict(validation['komentar_processed'])
            tuning_records.append({
                'experiment': arm,
                'features': feat_key,
                'architecture': arch_key,
                'hidden_layer_sizes': str(hidden_sizes),
                'fit_rows': len(fit_data),
                'fit_seconds': fit_seconds,
                **metrics(validation['label'], pred),
            })

tuning_results = pd.DataFrame(tuning_records)
ranked = tuning_results.sort_values(
    ['f1_1', 'macro_f1', 'experiment', 'features', 'architecture'],
    ascending=[False, False, True, True, True],
    kind='stable',
)
best_per_arm = ranked.groupby('experiment', sort=False).head(1).reset_index(drop=True)
selected_arm = str(ranked.iloc[0]['experiment'])
selected_config = ranked.iloc[0][['experiment', 'features', 'architecture', 'hidden_layer_sizes']].to_dict()

display(ranked.round(4))
print('best conf:')
display(best_per_arm.round(4))
print('selected conf', selected_config)

## 6. Refit conf for dev data

In [ ]:
development = pd.concat([train, validation], ignore_index=True)
dev_augmented = pd.concat([development[common_cols], synthetic[common_cols]], ignore_index=True)

final_models, fit_times = {}, {}
for row in best_per_arm.itertuples(index=False):
    fit_data = development if row.experiment == 'original_only' else dev_augmented
    hidden_sizes = ARCHITECTURES[row.architecture]
    model = make_pipeline(row.features, hidden_sizes)
    start = perf_counter()
    model.fit(fit_data['komentar_processed'], fit_data['label'])
    fit_times[row.experiment] = perf_counter() - start
    final_models[row.experiment] = model

selected_model = final_models[selected_arm]
print('done')

## 7. Evaluasi akhir

In [ ]:
test_records, predictions, probabilities, reports, matrices = [], {}, {}, {}, {}
for arm, model in final_models.items():
    start = perf_counter()
    pred = model.predict(test['komentar_processed'])
    prediction_seconds = perf_counter() - start
    prob = model.predict_proba(test['komentar_processed'])[:, 1]
    predictions[arm] = pred
    probabilities[arm] = prob
    test_records.append({
        'experiment': arm,
        **metrics(test['label'], pred),
        'refit_seconds': fit_times[arm],
        'test_predict_seconds': prediction_seconds,
    })
    reports[arm] = classification_report(
        test['label'], pred, labels=[0, 1], target_names=LABEL_NAMES, output_dict=True, zero_division=0
    )
    matrices[arm] = confusion_matrix(test['label'], pred, labels=[0, 1])
    print(arm)
    display(pd.DataFrame(reports[arm]).T.round(4))

dummy = DummyClassifier(strategy='most_frequent').fit(np.zeros((len(development), 1)), development['label'])
dummy_pred = dummy.predict(np.zeros((len(test), 1)))
test_records.append({
    'experiment': 'majority_baseline',
    **metrics(test['label'], dummy_pred),
    'refit_seconds': None,
    'test_predict_seconds': None,
})

test_results = pd.DataFrame(test_records)
display(test_results.round(4))
print('Pilihan model sebelum test:', selected_arm)

## 8. Confusion matrix dan visualisasi

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

fig_cm, axes = plt.subplots(1, 2, figsize=(10, 4.2), constrained_layout=True)
for ax, (arm, matrix) in zip(axes, matrices.items()):
    disp = ConfusionMatrixDisplay(confusion_matrix=matrix, display_labels=['0: nonjudi', '1: judi'])
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'ANN ({arm})')
fig_cm.savefig(OUT_DIR / 'confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

metric_cols = ['accuracy', 'precision_1', 'recall_1', 'f1_1', 'macro_f1']
fig_metrics, ax_m = plt.subplots(figsize=(8, 4.5), constrained_layout=True)
df_plot = test_results.set_index('experiment')[metric_cols]
df_plot.T.plot(kind='bar', ax=ax_m, rot=0)
ax_m.set_ylim(0, 1.08)
ax_m.set_ylabel('Score')
ax_m.set_title('Perbandingan Metrik Evaluasi Test - ANN')
ax_m.grid(axis='y', linestyle='--', alpha=0.5)
fig_metrics.savefig(OUT_DIR / 'test_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Analisis kesalahan prediksi

In [ ]:
test_predictions = test[['source_row', 'komentar', 'komentar_processed', 'label']].copy()
for arm in ['original_only', 'with_synthetic']:
    test_predictions[f'pred_{arm}'] = predictions[arm]
    test_predictions[f'prob_1_{arm}'] = probabilities[arm]

error_mask = test['label'] != predictions[selected_arm]
errors = test_predictions.loc[error_mask].copy()
errors['error_type'] = np.where(errors['label'] == 0, 'false_positive', 'false_negative')

print(f"Total kesalahan pada test ({selected_arm}): {len(errors)}")
display(errors['error_type'].value_counts())
print("\nContoh False Positives (nonjudi terprediksi judi):")
display(errors[errors['error_type'] == 'false_positive'][['komentar', 'prob_1_' + selected_arm]].head(5))
print("\nContoh False Negatives (judi terprediksi nonjudi):")
display(errors[errors['error_type'] == 'false_negative'][['komentar', 'prob_1_' + selected_arm]].head(5))

## 10. Bobot ANN

In [ ]:
tfidf_step = selected_model.named_steps['tfidf']
ann_step = selected_model.named_steps['ann']
vocab_series = pd.Series(tfidf_step.get_feature_names_out(), name='fitur')
coefs = ann_step.coefs_
proxy_weights = coefs[0] @ coefs[1].ravel()

feature_weights = pd.DataFrame({
    'fitur': vocab_series,
    'proxy_weight_1_vs_0': proxy_weights,
}).sort_values('proxy_weight_1_vs_0', ascending=False).reset_index(drop=True)

print("Top 10 fitur mengindikasikan promosi judi (bobot positif tertinggi):")
display(feature_weights.head(10))
print("\nTop 10 fitur mengindikasikan nonjudi (bobot negatif terendah):")
display(feature_weights.tail(10))

## 11. Demo

In [ ]:
url_pattern = re.compile(r'(?:https?://|www\.)[^\s<>]+', re.IGNORECASE)

def clean_comment(text):
    text = unicodedata.normalize('NFKC', text)
    text = ''.join('' if unicodedata.category(c) == 'Cf' else c for c in text)
    text = ''.join(' ' if unicodedata.category(c) == 'Cc' else c for c in text)
    text = url_pattern.sub('<url>', text.lower())
    return re.sub(r'\s+', ' ', text).strip()

new_samples = [
    'Konten review otomotifnya keren banget bang, ditunggu video berikutnya!',
    'Modal 20k bisa jp maxwin di situs SITUSGACOR77 deposit lancar kilat 🔥',
    'Hati-hati banyak bot penipuan dan link slot gacor di komentar',
    'Penjelasan mekanik mesinnya sangat detail dan mudah dipahami, terima kasih',
]

demo = pd.DataFrame({'komentar': new_samples})
demo['komentar_processed'] = demo['komentar'].map(clean_comment)
demo['prediksi'] = selected_model.predict(demo['komentar_processed'])
demo['prob_promosi_judi'] = selected_model.predict_proba(demo['komentar_processed'])[:, 1]
display(demo)

## 12. Save model dan artefak evaluasi

In [ ]:
# ngikuting yang ateng

OUT_DIR.mkdir(parents=True, exist_ok=True)

tuning_results.to_csv(OUT_DIR / 'validation_results.csv', index=False)
test_results.to_csv(OUT_DIR / 'test_metrics.csv', index=False)
test_predictions.to_csv(OUT_DIR / 'test_predictions.csv', index=False, encoding='utf-8-sig')
errors.to_csv(OUT_DIR / 'selected_model_errors.csv', index=False, encoding='utf-8-sig')
feature_weights.to_csv(OUT_DIR / 'selected_model_feature_weights.csv', index=False, encoding='utf-8-sig')

for arm, model in final_models.items():
    joblib_path = OUT_DIR / f'ann_{arm}.joblib'
    joblib.dump(model, joblib_path)
    restored = joblib.load(joblib_path)
    assert np.array_equal(restored.predict(test['komentar_processed']), predictions[arm])

summary = {
    'algorithm': 'Artificial Neural Network (MLPClassifier)',
    'selection_metric': 'validation positive-class F1, then macro-F1',
    'seed': SEED,
    'split_fractions': {'train': 1 - TEST_SIZE - VALIDATION_SIZE, 'validation': VALIDATION_SIZE, 'test': TEST_SIZE},
    'original_rows': len(original),
    'synthetic_train_rows': len(synthetic),
    'split_counts': {name: len(part) for name, part in parts.items()},
    'selected_experiment_before_test': selected_arm,
    'best_validation_per_experiment': best_per_arm.to_dict(orient='records'),
    'test_metrics': test_results.astype(object).where(test_results.notna(), None).to_dict(orient='records'),
    'classification_reports': reports,
    'confusion_matrices': {arm: matrix.tolist() for arm, matrix in matrices.items()},
    'versions': {
        'python': platform.python_version(),
        'sklearn': sklearn.__version__,
        'pandas': pd.__version__,
        'numpy': np.__version__,
        'joblib': joblib.__version__,
    },
    'model_input': 'komentar_processed; preprocessing is provided in notebook',
    'limitations': [
        'label meaning and annotation process unconfirmed',
        'random text split, not video/campaign holdout',
        'near-duplicate templates may span splits',
        'synthetic template shortcuts',
        'test has not been used to select parameters; avoid iterative tuning against test',
    ],
}
(OUT_DIR / 'experiment_summary.json').write_text(
    json.dumps(summary, ensure_ascii=False, indent=2, allow_nan=False),
    encoding='utf-8'
)

print('artefak eksperimen ANN berhasil disimpan ke:', OUT_DIR.relative_to(PROJECT_DIR))